In [13]:
import pandas as pd
import numpy as np
import os

# 1. Klasörü tara ve dosyaları akıllıca eşleştir
files = [f for f in os.listdir('.') if f.endswith(('.csv', '.xlsx'))]
gym_file = next((f for f in files if 'gym' in f.lower()), None)
injury_file = next((f for f in files if 'injury' in f.lower()), None)

if not gym_file or not injury_file:
    print("HATA: Dosyalar bulunamadı!")
    print(f"Klasördeki mevcut dosyalar: {files}")
else:
    try:
        print(f"Eşleşen Dosyalar:\n- Gym: {gym_file}\n- Injury: {injury_file}")
        
        # 2. Verileri Oku (CSV veya Excel ayrımı yapmadan)
        gym_df = pd.read_csv(gym_file)
        injury_df = pd.read_excel(injury_file) if injury_file.endswith('.xlsx') else pd.read_csv(injury_file)

        # 3. Veri Mühendisliği
        gym_df['Height_cm'] = gym_df['Height (m)'] * 100
        gym_df['Weight_kg'] = gym_df['Weight (kg)']

        df_final = pd.DataFrame()
        df_final['Yas'] = pd.concat([gym_df['Age'], injury_df['Age']], ignore_index=True)
        df_final['Boy'] = pd.concat([gym_df['Height_cm'], injury_df['Height_cm']], ignore_index=True)
        df_final['Kilo'] = pd.concat([gym_df['Weight_kg'], injury_df['Weight_kg']], ignore_index=True)

        # Cinsiyet: Erkek=0, Kadın=1
        gym_gender = gym_df['Gender'].map({'Male': 0, 'Female': 1})
        df_final['Cinsiyet'] = pd.concat([gym_gender, injury_df['Gender']], ignore_index=True)

        # 4. Eksik Özellikleri Rastgele/Mantıksal Doldur
        size = len(df_final)
        df_final['Aktivite_Seviyesi'] = np.random.choice([1.2, 1.375, 1.55, 1.725, 1.9], size)
        df_final['Sakatlik_Bolgesi'] = np.random.choice(['Yok', 'Diz', 'Omuz', 'Bel', 'Ayak Bileği'], size)
        df_final['Spor_Gecmisi_Yil'] = np.random.randint(0, 15, size)
        df_final['Ekipman'] = np.random.choice(['Full Gym', 'Dumbbell Only', 'Bodyweight'], size)
        df_final['Ogun_Tercihi'] = np.random.choice([3, 4, 5], size)
        df_final['VKI'] = (df_final['Kilo'] / ((df_final['Boy'] / 100) ** 2)).round(2)

        # Sakatlık Şiddeti (0-5 Arası)
        inj_risk = injury_df['Injury_Risk'] * 4.0 + np.random.uniform(0, 1, len(injury_df))
        df_final['Sakatlik_Siddeti'] = pd.concat([pd.Series(np.zeros(len(gym_df))), inj_risk], ignore_index=True).round(1)

        # 5. Program Etiketleme
        def labeler(row):
            if row['Sakatlik_Siddeti'] > 3.0: return 'REHABILITASYON', 'Sakatlık Sonrası'
            if row['VKI'] > 28.5: return 'YAG_YAKIMI', 'Kilo Verme'
            if row['VKI'] < 19.0: return 'KILO_ALMA', 'Kas Kazanımı'
            return 'PPL_SPLIT', 'Güç ve Kondisyon'

        labels = df_final.apply(labeler, axis=1)
        df_final['Program_Etiketi'] = [x[0] for x in labels]
        df_final['Hedef'] = [x[1] for x in labels]

        # 6. Kaydet ve Bitir
        cols = ['Yas', 'Cinsiyet', 'Boy', 'Kilo', 'Aktivite_Seviyesi', 'Sakatlik_Bolgesi', 
                'Sakatlik_Siddeti', 'Spor_Gecmisi_Yil', 'Hedef', 'Ekipman', 'Ogun_Tercihi', 'VKI', 'Program_Etiketi']
        
        df_final[cols].dropna().to_csv('personel_ai_veri_seti.csv', index=False)
        print(f"\nİşlem Başarılı! {len(df_final)} satır kaydedildi.")

    except Exception as e:
        print(f"Hata: {e}")

Eşleşen Dosyalar:
- Gym: gym_members_exercise_tracking.csv
- Injury: High_Accuracy_Sport_Injury_Dataset.xlsx

İşlem Başarılı! 1573 satır kaydedildi.
